# Mini Project: Policy Q&A Assistant (Day 3, teams of 4 to 5)
**Time:** 2 hours including a 5-minute demo per team.

Your team builds one small assistant for Meridian Retail employees by combining what you did in Labs 5 to 9:
1. **Knowledge:** answers from the policy documents, with citations (Labs 5, 6)
2. **Action:** one tool that raises an HR or IT ticket, behind an approval step (Labs 7, 8)
3. **Quality:** a test score on at least 8 questions (Lab 9)

Every step already works with sensible defaults. Your job is to make **decisions** (marked `TEAM DECISION`)
and be ready to explain them in the demo. Assign roles: one driver at the keyboard, one note-taker for the
demo, the rest review each decision.

In [ ]:
# SETUP: run this cell first. It works in Google Colab and in VS Code.
import os, sys, subprocess
REPO_URL = "https://github.com/cybertide-solutions/ai-platform-architect-labs.git"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/course"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/course"], check=True)
    os.chdir("/content/course")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                      # VS Code starts in notebooks/; the course files are one level up
sys.path.insert(0, os.getcwd())
import labkit as lk
lk.setup();
import json, datetime

## Step 1: Knowledge layer
`TEAM DECISION`: chunk size and overlap. Use what you learned in Lab 6.

In [ ]:
CHUNK_SIZE, OVERLAP = 600, 100          # TEAM DECISION
qc, chunks = lk.build_policy_index(size=CHUNK_SIZE, overlap=OVERLAP)
print(len(chunks), "chunks")

## Step 2: Answer with citations
`TEAM DECISION`: the system prompt. Start from `lk.RAG_SYSTEM` and adapt the tone and rules for employees.

In [ ]:
SYSTEM = lk.RAG_SYSTEM + "\nAddress the employee politely. If a policy needs an approval, say who approves."  # TEAM DECISION

def answer(question: str) -> dict:
    return lk.rag_answer(question, qc, k=4, system=SYSTEM)

r = answer("How many sick leave days do I get, and when do I need a certificate?")
print(r["answer"], "\nSOURCES:", r["sources"])

## Step 3: One action tool, behind approval
Employees can ask the assistant to raise a ticket. `TEAM DECISION`: which tickets need human approval?

In [ ]:
TICKETS = []

def raise_ticket(team: str, summary: str, urgency: str) -> dict:
    """Raise a ticket with HR, IT or Finance."""
    t = {"id": f"TCK-{len(TICKETS)+1:04d}", "team": team, "summary": summary, "urgency": urgency,
         "created": datetime.datetime.now().isoformat(timespec="seconds")}
    TICKETS.append(t)
    return t

def search_policies(query: str) -> list:
    return [{"doc": h["doc"], "page": h["page"], "text": h["text"]} for h in lk.search(qc, query, k=3)]

TOOLS = {
    "search_policies": (search_policies, lk.tool_spec(search_policies, "Search Meridian company policies.",
                        {"query": {"type": "string"}})),
    "raise_ticket": (raise_ticket, lk.tool_spec(raise_ticket, "Raise a ticket with HR, IT or Finance.",
                     {"team": {"type": "string", "enum": ["HR", "IT", "Finance"]},
                      "summary": {"type": "string"},
                      "urgency": {"type": "string", "enum": ["low", "medium", "high"]}})),
}

def approval_rule(name: str, args: dict) -> bool:       # TEAM DECISION
    return name == "raise_ticket" and args.get("urgency") == "high"

def approver(name, args):
    return (not approval_rule(name, args)) or lk.console_approver(name, args)

AGENT_SYSTEM = ("You are Meridian's employee assistant. Search the policies before answering policy questions "
                "and cite the document. Raise a ticket only when the employee asks for help that needs a person.")

def assistant(message: str) -> str:
    return lk.run_agent(message, TOOLS, system=AGENT_SYSTEM, needs_approval={"raise_ticket"}, approver=approver)

assistant("My laptop was stolen from my car this morning. What do I need to do? Please raise a ticket with IT.")
print("Tickets:", TICKETS)

## Step 4: Measure quality
The first 5 test questions are provided. `TEAM DECISION`: add at least 3 of your own, including one the
documents cannot answer.

In [ ]:
tests = lk.load_csv("data/eval_starter.csv")
tests += [
    {"question": "How long is paternity leave?", "expected_answer": "4 weeks", "source_doc": "01_leave_policy.pdf"},
    # TEAM: add your questions here, for example:
    # {"question": "...", "expected_answer": "...", "source_doc": "0X_....pdf"},
    # {"question": "Who is the CEO of Meridian?", "expected_answer": "NOT IN DOCUMENTS", "source_doc": "none"},
]
scores = lk.evaluate(qc, tests, system=SYSTEM, label="team")
scores[["question", "retrieval_hit", "verdict", "judge_reason"]]

## Step 5: Demo checklist (5 minutes per team)
Fill this in, then show it on screen.
* **Our design decisions:** chunk size ___ because ___; approval rule ___ because ___
* **Score:** ___ of ___ questions passed. The failure we learned most from: ___
* **Live demo:** one policy question with citations, one ticket request that triggers approval
* **What we would do before production:** ___